In [ ]:
"""
Interactive CORD-19 Summarization Application

This script creates a simple Streamlit application that demonstrates the
CORD-19 Summarization and Keyword Extraction Agent.
"""

import streamlit as st
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from cord19_summarizer import CORD19Agent
import os
import nltk
import plotly.express as px
from wordcloud import WordCloud

# Download NLTK resources
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)

# Page configuration
st.set_page_config(
    page_title="CORD-19 Summarization Agent",
    page_icon="🧬",
    layout="wide"
)

@st.cache_resource
def load_agent():
    """Create and return the CORD-19 agent (cached)."""
    agent = CORD19Agent()
    return agent

@st.cache_data
def download_data(_agent):
    """Download the dataset using the agent (cached)."""
    data_path = _agent.download_dataset()
    return data_path

@st.cache_data
def load_metadata(_agent, data_path):
    """Load the metadata using the agent (cached)."""
    metadata = _agent.load_metadata(os.path.join(data_path, "metadata.csv"))
    return metadata

def create_wordcloud(text):
    """Create a word cloud from text."""
    wordcloud = WordCloud(
        width=800, 
        height=400, 
        background_color='white',
        max_words=100,
        colormap='viridis',
        collocations=False
    ).generate(text)
    
    return wordcloud

def main():
    st.title("🧬 CORD-19 Research Summarization Agent")
    st.write("""
    This application uses TensorFlow and vector storage to analyze the CORD-19 dataset,
    generate summaries, extract keywords, and find similar research papers.
    """)
    
    # Initialize the agent
    with st.spinner("Loading models..."):
        agent = load_agent()
    
    # Sidebar for navigation
    st.sidebar.title("Navigation")
    page = st.sidebar.radio(
        "Select a page",
        ["Data Explorer", "Summarization", "Keyword Analysis", "Similar Papers Search"]
    )
    
    # Download and load data
    with st.spinner("Downloading and preparing dataset..."):
        data_path = download_data(agent)
        metadata = load_metadata(agent, data_path)
    
    # Data Explorer
    if page == "Data Explorer":
        st.header("📊 Data Explorer")
        st.write(f"Dataset contains {len(metadata)} research papers")
        
        # Display sample data
        st.subheader("Sample Data")
        st.dataframe(metadata.head(10))
        
        # Basic statistics
        st.subheader("Dataset Statistics")
        col1, col2 = st.columns(2)
        
        # Publication year distribution
        with col1:
            if 'publish_time' in metadata.columns:
                # Extract year from publish_time
                metadata['year'] = pd.to_datetime(metadata['publish_time'], errors='coerce').dt.year
                year_counts = metadata['year'].value_counts().sort_index()
                
                # Create interactive plot
                fig = px.bar(
                    x=year_counts.index, 
                    y=year_counts.values,
                    labels={'x': 'Publication Year', 'y': 'Number of Papers'},
                    title='Papers by Publication Year'
                )
                st.plotly_chart(fig, use_container_width=True)
            else:
                st.write("Publication date information not available")
        
        # Authors per paper distribution
        with col2:
            if 'authors' in metadata.columns:
                # Count authors per paper
                metadata['author_count'] = metadata['authors'].apply(
                    lambda x: len(str(x).split(';')) if pd.notna(x) else 0
                )
                
                # Create histogram
                fig = px.histogram(
                    metadata, 
                    x='author_count',
                    nbins=20,
                    range_x=[0, 20],
                    labels={'author_count': 'Number of Authors', 'count': 'Number of Papers'},
                    title='Authors per Paper Distribution'
                )
                st.plotly_chart(fig, use_container_width=True)
            else:
                st.write("Author information not available")
        
        # Abstract availability
        st.subheader("Abstract Availability")
        abstract_available = metadata['abstract'].notna().sum()
        abstract_missing = len(metadata) - abstract_available
        
        fig = px.pie(
            values=[abstract_available, abstract_missing],
            names=['Available', 'Missing'],
            title='Abstract Availability'
        )
        st.plotly_chart(fig, use_container_width=True)
    
    # Summarization
    elif page == "Summarization":
        st.header("📝 Paper Summarization")
        st.write("""
        Generate summaries of research papers using the T5 transformer model.
        """)
        
        # Sample papers for summarization
        if st.button("Generate Sample Summaries"):
            with st.spinner("Generating document embeddings and summaries..."):
                # Create document embeddings if not already created
                documents, _ = agent.create_document_embeddings()
                
                # Generate summaries for a sample of documents
                summaries = agent.generate_batch_summaries(documents, sample_size=5)
                
                # Display summaries
                for i, summary in enumerate(summaries):
                    st.subheader(f"Paper {i+1}: {summary['title']}")
                    st.write(f"**Summary:** {summary['summary']}")
                    st.write("---")
        
        # Custom paper summarization
        st.subheader("Summarize Custom Text")
        custom_text = st.text_area(
            "Enter abstract text to summarize:",
            height=200,
            placeholder="Paste a research paper abstract here..."
        )
        
        if st.button("Generate Summary") and custom_text:
            with st.spinner("Generating summary..."):
                summary = agent.generate_summary(custom_text)
                st.write("**Summary:**")
                st.info(summary)
    
    # Keyword Analysis
    elif page == "Keyword Analysis":
        st.header("🔑 Keyword Analysis")
        st.write("""
        Extract key topics and keywords from the CORD-19 dataset using NMF on TF-IDF vectors.
        """)
        
        if st.button("Extract Keywords"):
            with st.spinner("Creating document embeddings and extracting keywords..."):
                # Create document embeddings if not already created
                documents, _ = agent.create_document_embeddings()
                
                # Extract topics/keywords
                topics = agent.extract_keywords(documents, num_topics=5, num_words=15)
                
                # Display topics/keywords
                col1, col2 = st.columns(2)
                
                with col1:
                    st.subheader("Topics and Keywords")
                    for topic in topics:
                        st.write(f"**Topic {topic['topic_id']+1}:** {', '.join(topic['words'])}")
                
                with col2:
                    st.subheader("Word Cloud")
                    # Create a text string with all keywords
                    all_keywords = ' '.join([' '.join(topic['words']) for topic in topics])
                    wordcloud = create_wordcloud(all_keywords)
                    
                    # Display word cloud
                    fig, ax = plt.subplots(figsize=(10, 5))
                    ax.imshow(wordcloud, interpolation='bilinear')
                    ax.axis('off')
                    st.pyplot(fig)
    
    # Similar Papers Search
    elif page == "Similar Papers Search":
        st.header("🔍 Similar Papers Search")
        st.write("""
        Find papers similar to a query using vector similarity search.
        """)
        
        # Input for search query
        query = st.text_input(
            "Enter search query:",
            placeholder="e.g., covid-19 transmission in indoor environments"
        )
        
        k = st.slider("Number of results to display", min_value=1, max_value=10, value=5)
        
        if st.button("Search") and query:
            with st.spinner("Searching for similar papers..."):
                # Make sure vectors are created
                try:
                    # Try to load existing vector store
                    similar_docs = agent.search_similar_documents(query, k=k)
                except:
                    # Create document embeddings if not already created
                    documents, _ = agent.create_document_embeddings()
                    similar_docs = agent.search_similar_documents(query, k=k)
                
                # Display results
                if similar_docs:
                    for i, doc in enumerate(similar_docs):
                        st.subheader(f"Match {i+1} (Score: {doc['similarity_score']:.4f})")
                        st.write(f"**Title:** {doc['metadata'].get('title', 'N/A')}")
                        st.write(f"**Content:** {doc['content'][:300]}...")
                        st.write("---")
                else:
                    st.warning("No similar papers found or vector store not created.")

if __name__ == "__main__":
    main()